# Aircraft Population Analysis Utilities

Shared derivation and aggregation functions for the aircraft population report suite.

In [ ]:
import numpy as np
import pandas as pd

AGE_PERCENTILES = [0.10, 0.25, 0.50, 0.75, 0.90, 0.95]
ERA_ORDER = ['Before 1950', '1950-1969', '1970-1989', '1990-2009', '2010 onwards', 'Unknown']


def load_aircraft_population():
    """Load the read-only observation and enrichment population dataset.

    :return: A dataframe containing one row per tracked-aircraft observation record.
    """
    query = construct_query('tracker', 'reports', 'aircraft-population.sql', {})
    population = query_data('tracker', query)
    for column in ['Session Started At UTC', 'First Observation', 'Last Observation']:
        population[column] = pd.to_datetime(population[column], utc=True, errors='coerce')
    population['Manufacture Year'] = pd.to_numeric(population['Manufacture Year'], errors='coerce').astype('Int64')
    population['Age At Observation'] = pd.to_numeric(population['Age At Observation'], errors='coerce').astype('Int64')
    population['Session Period'] = np.where(population['Session Started At UTC'].dt.hour.between(6, 17), 'Day', 'Evening')
    population['Manufacturing Era'] = manufacturing_era(population['Manufacture Year'])
    return population


def manufacturing_era(years):
    """Classify manufacture years into the eras specified by the project brief.

    :param years: A pandas series containing nullable manufacture years.
    :return: An ordered categorical series of manufacturing eras.
    """
    values = pd.cut(years.astype(float), bins=[-np.inf, 1949, 1969, 1989, 2009, np.inf], labels=ERA_ORDER[:-1])
    return values.astype(object).fillna('Unknown').astype(pd.CategoricalDtype(ERA_ORDER, ordered=True))


def unique_aircraft(population):
    """Return one representative row per ICAO address without inflating fleet counts.

    :param population: Observation-level population dataframe.
    :return: A dataframe containing the latest observation of each aircraft.
    """
    return (population.sort_values('First Observation')
            .drop_duplicates('Address', keep='last')
            .reset_index(drop=True))


def coverage_summary(population):
    """Summarise manufacture-year enrichment coverage for unique aircraft.

    :param population: Observation-level population dataframe.
    :return: A one-row dataframe with known, missing, and percentage coverage values.
    """
    aircraft = unique_aircraft(population)
    known = int(aircraft['Manufacture Year'].notna().sum())
    total = len(aircraft)
    return pd.DataFrame([{'Unique Aircraft': total, 'Known Manufacture Year': known,
                          'Missing Manufacture Year': total - known,
                          'Known Coverage %': round(100 * known / total, 1) if total else 0.0}])


def grouped_age_summary(population, dimension):
    """Build observation, aircraft, and age measures for a categorical dimension.

    :param population: Observation-level population dataframe.
    :param dimension: Column used to group the population.
    :return: A dataframe sorted by median age and unique-aircraft count.
    """
    rows = []
    for label, group in population.groupby(dimension, observed=False, dropna=False):
        known = group.dropna(subset=['Age At Observation'])
        oldest = known.sort_values('Age At Observation', ascending=False).head(1)
        rows.append({dimension: label, 'Observations': len(group), 'Unique Aircraft': group['Address'].nunique(),
                     'Known Age Observations': len(known), 'Median Age': known['Age At Observation'].median(),
                     'Oldest Age': known['Age At Observation'].max(),
                     'Oldest Example': oldest['Registration'].iloc[0] if len(oldest) else 'Unavailable'})
    return pd.DataFrame(rows).sort_values(['Median Age', 'Unique Aircraft'], ascending=[False, False], na_position='last')


def aircraft_frequency(population):
    """Aggregate observation and session frequency for each aircraft.

    :param population: Observation-level population dataframe.
    :return: A dataframe with one row per ICAO address.
    """
    identity = unique_aircraft(population).set_index('Address')
    frequency = population.groupby('Address').agg(Observations=('Observation Id', 'count'), Sessions=('Session Id', 'nunique'))
    columns = ['Registration', 'Manufacture Year', 'Age At Observation', 'ICAO Type', 'Model', 'Manufacturer']
    return identity[columns].join(frequency).reset_index()


def rarity_candidates(population):
    """Calculate transparent discovery signals without assigning importance.

    :param population: Observation-level population dataframe.
    :return: Aircraft-level candidates with reasons that can be interactively filtered.
    """
    aircraft = aircraft_frequency(population)
    model_counts = aircraft.groupby('Model')['Address'].transform('count')
    manufacturer_counts = aircraft.groupby('Manufacturer')['Address'].transform('count')
    aircraft['Model Aircraft Count'] = model_counts
    aircraft['Manufacturer Aircraft Count'] = manufacturer_counts
    aircraft['Candidate Reasons'] = aircraft.apply(lambda row: '; '.join([
        reason for condition, reason in [(pd.notna(row['Age At Observation']) and row['Age At Observation'] >= 60, 'age >= 60'),
                                         (row['Model'] != 'Unknown' and row['Model Aircraft Count'] == 1, 'model occurs once'),
                                         (row['Manufacturer'] != 'Unknown' and row['Manufacturer Aircraft Count'] == 1, 'manufacturer occurs once'),
                                         (row['Sessions'] == 1, 'single session')]
        if condition]), axis=1)
    return aircraft.sort_values(['Age At Observation', 'Sessions'], ascending=[False, True], na_position='last')
